# Feature engineering for community crime classification

This notebook transforms raw crime statistics and census demographics into a
community-level feature matrix with a `risk_level` target label (Low / Medium /
High). We create risk labels from crime-count percentiles, scale features,
handle missing values, and explore class distributions and correlations.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from src.data_loader import (
    load_or_fetch_crime_data, load_or_fetch_census_data,
    preprocess_crime_data, preprocess_census_data,
    create_community_features,
)
from src.model import create_risk_labels, CRIME_RISK_BINS

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load raw crime and census data

In [ ]:
crime_raw = load_or_fetch_crime_data(data_dir='../data', limit=100000)
print(f'Crime records: {len(crime_raw):,}')
print(f'Columns: {list(crime_raw.columns)}')

In [ ]:
census_raw = load_or_fetch_census_data(data_dir='../data', limit=100000)
print(f'Census records: {len(census_raw):,}')

## 2. Preprocess and build community-level features

`preprocess_crime_data` converts `crime_count`, `year`, and `month` to numeric
types and drops rows with missing critical fields. `create_community_features`
aggregates crimes per community, pivots by crime category, and optionally merges
census demographics to compute per-capita crime rates.

In [ ]:
crime_df = preprocess_crime_data(crime_raw)
census_df = preprocess_census_data(census_raw)

community_df = create_community_features(crime_df, census_df)
print(f'Communities: {len(community_df)}')
print(f'Feature columns: {community_df.shape[1]}')
community_df.head()

## 3. Create risk labels from percentiles

`create_risk_labels` in `model.py` assigns each community a risk level based
on percentile rank of `total_crimes`: bottom third is Low, middle third is
Medium, top third is High.

In [ ]:
print(f'Risk bin thresholds: {CRIME_RISK_BINS}')

community_df = create_risk_labels(community_df, column='total_crimes')
print(f'\nRisk level distribution:')
print(community_df['risk_level'].value_counts().to_string())

In [ ]:
# Visualise class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

order = ['Low', 'Medium', 'High']
community_df['risk_level'].value_counts().reindex(order).plot(
    kind='bar', ax=axes[0], color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='black'
)
axes[0].set_title('Class distribution of risk_level', fontsize=11)
axes[0].set_ylabel('Number of communities')
axes[0].tick_params(axis='x', rotation=0)

# Total crimes distribution by risk level
for level, color in zip(order, ['#2ecc71', '#f39c12', '#e74c3c']):
    subset = community_df[community_df['risk_level'] == level]
    axes[1].hist(subset['total_crimes'], bins=20, alpha=0.6, label=level, color=color)
axes[1].set_title('Total crimes by risk level', fontsize=11)
axes[1].set_xlabel('Total crimes')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Handle missing values

Census data may not be available for every community (industrial zones,
new developments). We inspect missingness and fill numeric NaNs with zeros
where appropriate.

In [ ]:
missing_pct = community_df.isnull().mean().sort_values(ascending=False) * 100
print('Columns with missing values (%):\n')
print(missing_pct[missing_pct > 0].round(1).to_string())

In [ ]:
# Fill missing demographic and crime-rate columns with 0
fill_cols = [c for c in community_df.select_dtypes(include=[np.number]).columns
             if community_df[c].isnull().any()]
for col in fill_cols:
    community_df[col] = community_df[col].fillna(0)
print(f'Filled {len(fill_cols)} columns with zeros')
print(f'Remaining NaN count: {community_df.isnull().sum().sum()}')

## 5. Feature scaling with StandardScaler

Logistic Regression and other distance-based methods are sensitive to feature
scale. We fit a `StandardScaler` on the numeric columns and inspect the result.

In [ ]:
exclude_cols = ['community', 'code', 'risk_level', 'total_crimes']
numeric_cols = [c for c in community_df.select_dtypes(include=[np.number]).columns
                if c not in exclude_cols]

scaler = StandardScaler()
scaled_values = scaler.fit_transform(community_df[numeric_cols])
scaled_df = pd.DataFrame(scaled_values, columns=numeric_cols, index=community_df.index)

print(f'Scaled {len(numeric_cols)} features')
print(f'Mean after scaling (should be ~0): {scaled_df.mean().mean():.4f}')
print(f'Std after scaling (should be ~1):  {scaled_df.std().mean():.4f}')
scaled_df.describe().round(2)

## 6. Correlation analysis

We examine correlations among crime category counts and demographic features
to understand which variables carry the most signal for risk classification.

In [ ]:
corr = community_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            ax=ax, linewidths=0.3)
ax.set_title('Feature correlation matrix (community-level)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Identify highly correlated pairs (|r| > 0.9)
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.9:
            high_corr.append((corr.columns[i], corr.columns[j], round(r, 3)))

if high_corr:
    print('Highly correlated pairs (|r| > 0.9):\n')
    for a, b, r in high_corr:
        print(f'  {a}  <->  {b}  :  r = {r}')
else:
    print('No pairs with |r| > 0.9')

## 7. Save the engineered dataset

In [ ]:
import os
os.makedirs('../data', exist_ok=True)
community_df.to_csv('../data/community_features_engineered.csv', index=False)
print(f'Saved: {len(community_df)} communities, {community_df.shape[1]} columns')

## Summary

- Built community-level features from raw crime statistics and census data.
- Created `risk_level` labels using percentile-based binning (Low / Medium / High).
- Classes are roughly balanced by design (thirds of the community distribution).
- Filled missing census-linked columns with zeros and scaled all numeric features.
- Correlation analysis revealed that certain crime categories (e.g., theft,
  social disorder) are strongly correlated -- tree-based models can handle this,
  but Logistic Regression may benefit from dropping one of each correlated pair.

Next: `03_modeling` trains five classifiers on this feature set.